# Bronze Layer — Raw CSV ➜ Delta Ingestion

**SariMart Express — Automated Replenishment Demo (Microsoft Fabric)**

This notebook implements the **Bronze layer** of a medallion architecture. It reads every
source table as a raw **CSV** file from the Lakehouse landing zone (`Files/raw/`) and writes it
straight to a **Delta** table — **no transformation, no cleansing, schema preserved as-is**.

It ingests all 8 generated tables (7 dimensions + 1 fact) that feed the three business use cases:

| Use Case | Bronze tables it relies on |
|----------|----------------------------|
| **UC1 — Reorder Alerting** | `fact_sales`, `dim_product`, `dim_store`, `dim_supplier` |
| **UC2 — Stockout-Risk Heat Map** | `fact_sales`, `dim_store` |
| **UC3 — Overstock Detection** | `fact_sales`, `dim_product`, `dim_category` |

> **Prerequisite:** attach the `LH_Retail` Lakehouse (Explorer ➜ Add ➜ Existing Lakehouse), then **Run all**.

In [ ]:
# 1. Bronze layer configuration
RAW_PATH = "Files/landing_zone"   # landing zone for source CSV files
BRONZE_SCHEMA = "bronzedb"        # Delta tables are written to this schema

# All tables produced by the data generator (7 dimensions + 1 fact)
TABLES = [
    "dim_date", "dim_store", "dim_category", "dim_supplier",
    "dim_product", "dim_promotion", "dim_channel", "fact_sales",
]

def bronze_table_name(source_table):
    # Convert source names like dim_date -> date_dim and fact_sales -> sales_dim
    if source_table.startswith("dim_") or source_table.startswith("fact_"):
        base = source_table.split("_", 1)[1]
    else:
        base = source_table
    return f"{base}_dim"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
print(f"{len(TABLES)} tables to ingest into schema '{BRONZE_SCHEMA}'")

In [ ]:
# 2. OPTIONAL — land source CSVs into Files/landing_zone for a self-contained demo.
#    In production these CSVs arrive from the source system, so SKIP this cell.
#    Here we export generated tables to CSV once, then bronze reads them back.
import os
os.makedirs("/lakehouse/default/Files/landing_zone", exist_ok=True)
for t in TABLES:
    spark.table(t).toPandas().to_csv(f"/lakehouse/default/Files/landing_zone/{t}.csv", index=False)
    print(f"landed {t}.csv")

In [ ]:
# 3. Bronze ingestion — read landing-zone CSVs and write Delta tables to bronzedb
def ingest_to_bronze(table_name):
    src = f"{RAW_PATH}/{table_name}.csv"
    df = (spark.read.format("csv")
              .option("header", "true")
              .option("inferSchema", "true")
              .load(src))

    target_table = bronze_table_name(table_name)
    target = f"{BRONZE_SCHEMA}.{target_table}"

    (df.write
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .format("delta")
       .saveAsTable(target))

    print(f"✓ {src} -> {target} ({df.count():,} rows, {len(df.columns)} cols)")

for t in TABLES:
    ingest_to_bronze(t)

In [ ]:
# 4. Validate the bronze layer
print("Bronze tables loaded:\n")
for t in TABLES:
    bt = f"{BRONZE_SCHEMA}.{bronze_table_name(t)}"
    print(f"  {bt:30s} {spark.table(bt).count():>8,} rows")

print("\nPreview of bronzedb.sales_dim:")
spark.table(f"{BRONZE_SCHEMA}.{bronze_table_name('fact_sales')}").show(5)

## Notes

- **Raw fidelity:** the bronze layer is a faithful copy of the source CSVs — no renaming, typing,
  filtering, or business logic. Keys and columns are preserved exactly as generated.
- **Naming:** tables are written as `bronze_<table>`. If your Lakehouse is **schema-enabled**, you can
  instead write to a `bronze` schema (e.g., `bronze.dim_date`) by adjusting `saveAsTable`.
- **Idempotent:** `mode("overwrite")` makes re-runs safe; each run fully refreshes the bronze tables.
- **Next layer:** the **Silver** notebook would apply typing, de-duplication, and conformed keys, and
  **Gold** would compute the replenishment recommendations powering UC1, UC2, and UC3.